# LIMUC CLIP image embeddings + Linear Classifier
Extract CLIP image embeddings and train a linear classifier.


In [1]:
import os
import json
import random
from pathlib import Path
from typing import Tuple

import numpy as np
import pandas as pd
from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader

from transformers import CLIPModel, CLIPProcessor
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline


2026-02-28 20:14:56.585830: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-02-28 20:14:56.585866: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-02-28 20:14:56.645697: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-02-28 20:14:56.775051: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-28 20:14:57.937220: W tensorflow/compiler/tf2

In [2]:
# Paths & config
def find_dataset_root() -> Path:
    start = Path.cwd().resolve()
    for p in [start] + list(start.parents):
        if (p / "0_dataset_prep").exists():
            return p
    raise RuntimeError(f"Could not locate dataset root containing '0_dataset_prep'. cwd={start}")

DATA_ROOT = find_dataset_root()
META_CSV = DATA_ROOT / "0_dataset_prep" / "out" / "metadata" / "metadata_enriched.csv"
LABEL_MAP_CSV = DATA_ROOT / "0_dataset_prep" / "out" / "metadata" / "label_map.csv"
OUT_DIR = DATA_ROOT / "1_frozen_encoders" / "results" / "clip_linear_baseline"
OUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = os.getenv("CLIP_MODEL", "openai/clip-vit-base-patch32")
SEED = int(os.getenv("SEED", "42"))
BATCH_SIZE = int(os.getenv("BATCH_SIZE", "16"))
NUM_WORKERS = int(os.getenv("NUM_WORKERS", "0"))
MAX_SAMPLES = int(os.getenv("MAX_SAMPLES", "0")) or None
CACHE_FEATURES = os.getenv("CACHE_FEATURES", "1") == "1"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", DEVICE)
print("Data root:", DATA_ROOT)
print("Output dir:", OUT_DIR)



Device: cuda
Data root: /home/aristotle/Desktop/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/LIMUC
Output dir: /home/aristotle/Desktop/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/LIMUC/1_frozen_encoders/results/clip_linear_baseline


In [3]:
# Seed
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Load metadata
meta = pd.read_csv(META_CSV)

# Normalize image paths to absolute (base is 0_dataset_prep)
images_base = DATA_ROOT / "0_dataset_prep"

def to_abs(p):
    p = Path(p)
    if p.is_absolute():
        return p
    return (images_base / p).resolve()

meta["image_path"] = meta["image_path"].apply(lambda p: str(to_abs(p)))

# Label map
if LABEL_MAP_CSV.exists():
    label_map = pd.read_csv(LABEL_MAP_CSV)
    id_to_name = dict(zip(label_map.label_id, label_map.label_name))
else:
    id_to_name = {i: name for i, name in enumerate(sorted(meta.label_name.unique()))}

name_to_id = {v: k for k, v in id_to_name.items()}
meta["label_id"] = meta["label_name"].map(name_to_id)

# Keep only rows with existing images
meta = meta[meta["image_path"].apply(lambda p: Path(p).exists())].reset_index(drop=True)

if MAX_SAMPLES:
    meta = meta.sample(n=min(MAX_SAMPLES, len(meta)), random_state=SEED).reset_index(drop=True)

print("Rows:", len(meta))
print("Splits:\n", meta["split"].value_counts())

FileNotFoundError: [Errno 2] No such file or directory: '/home/aristotle/Desktop/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/LIMUC/0_dataset_prep/out/metadata/metadata_enriched.csv'

In [ ]:
class ImageDS(Dataset):
    def __init__(self, df: pd.DataFrame, processor):
        self.paths = df["image_path"].tolist()
        self.labels = df["label_id"].tolist()
        self.processor = processor

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        out = self.processor(images=img, return_tensors="pt")
        pixel_values = out["pixel_values"].squeeze(0)
        label = int(self.labels[idx])
        return pixel_values, label


In [ ]:
processor = CLIPProcessor.from_pretrained(MODEL_NAME)
model = CLIPModel.from_pretrained(MODEL_NAME).to(DEVICE)
model.eval()

def extract_split(df: pd.DataFrame) -> Tuple[np.ndarray, np.ndarray]:
    ds = ImageDS(df, processor)
    dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    feats, labels = [], []
    with torch.no_grad():
        for x, y in dl:
            x = x.to(DEVICE)
            out = model.get_image_features(pixel_values=x)
            out = out / out.norm(p=2, dim=-1, keepdim=True)
            feats.append(out.cpu().numpy())
            labels.append(y.numpy())
    return np.concatenate(feats, axis=0), np.concatenate(labels, axis=0)


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


ValueError: Due to a serious vulnerability issue in `torch.load`, even with `weights_only=True`, we now require users to upgrade torch to at least v2.6 in order to use the function. This version restriction does not apply when loading files with safetensors.
See the vulnerability report here https://nvd.nist.gov/vuln/detail/CVE-2025-32434

In [ ]:
# Extract / cache features
def _load_or_extract(split_name: str, df: pd.DataFrame):
    feat_path = OUT_DIR / f"features_{split_name}.npy"
    label_path = OUT_DIR / f"labels_{split_name}.npy"
    if CACHE_FEATURES and feat_path.exists() and label_path.exists():
        X = np.load(feat_path)
        y = np.load(label_path)
        return X, y
    X, y = extract_split(df)
    if CACHE_FEATURES:
        np.save(feat_path, X)
        np.save(label_path, y)
    return X, y

train_df = meta[meta["split"] == "train"].reset_index(drop=True)
val_df = meta[meta["split"].isin(["val", "validation"])].reset_index(drop=True)
test_df = meta[meta["split"] == "test"].reset_index(drop=True)

X_train, y_train = _load_or_extract("train", train_df)
X_val, y_val = _load_or_extract("val", val_df)
X_test, y_test = _load_or_extract("test", test_df)



In [ ]:
# Train linear classifier
clf = Pipeline([
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        n_jobs=-1,
        multi_class="multinomial",
    )),
])

clf.fit(X_train, y_train)

train_pred = clf.predict(X_train)
val_pred = clf.predict(X_val)
test_pred = clf.predict(X_test)

train_prob = clf.predict_proba(X_train)
val_prob = clf.predict_proba(X_val)
test_prob = clf.predict_proba(X_test)


/home/arcturus/miniforge3/envs/vqa-rag/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1264: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


In [ ]:
# =====================
# Metrics helpers
# =====================
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    cohen_kappa_score,
    mean_absolute_error,
    mean_squared_error,
    roc_auc_score,
)

try:
    from scipy.stats import spearmanr
    _HAS_SCIPY = True
except Exception:
    _HAS_SCIPY = False


def expected_calibration_error(y_true, y_prob, n_bins=10):
    if y_prob is None:
        return None
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    confidences = y_prob.max(axis=1)
    predictions = y_prob.argmax(axis=1)
    accuracies = (predictions == y_true).astype(float)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        mask = (confidences > bins[i]) & (confidences <= bins[i + 1])
        if mask.any():
            ece += abs(accuracies[mask].mean() - confidences[mask].mean()) * mask.mean()
    return float(ece)


def compute_metrics(y_true, y_pred, labels, label_names, y_prob=None):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    report = classification_report(
        y_true,
        y_pred,
        labels=labels,
        target_names=label_names,
        output_dict=True,
        zero_division=0,
    )

    summary = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro")),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted")),
        "qwk": float(cohen_kappa_score(y_true, y_pred, weights="quadratic")),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
    }

    if _HAS_SCIPY:
        summary["spearman"] = float(spearmanr(y_true, y_pred).correlation)

    if y_prob is not None:
        try:
            summary["auroc_ovr"] = float(roc_auc_score(y_true, y_prob, multi_class="ovr"))
        except Exception:
            summary["auroc_ovr"] = None
        summary["ece"] = expected_calibration_error(y_true, y_prob, n_bins=10)

    return summary, report


def save_split_outputs(
    split_name,
    y_true,
    y_pred,
    labels,
    label_names,
    out_dir,
    y_prob=None,
    df_meta=None,
):
    summary, report = compute_metrics(y_true, y_pred, labels, label_names, y_prob)

    # Save metrics
    metrics = {
        "split": split_name,
        "summary": summary,
        "report": report,
    }
    with open(out_dir / f"metrics_{split_name}.json", "w") as f:
        json.dump(metrics, f, indent=2)

    # Save per-class report
    per_class = {k: v for k, v in report.items() if k in label_names}
    pd.DataFrame(per_class).T.to_csv(out_dir / f"per_class_{split_name}.csv")

    # Save predictions
    pred_df = pd.DataFrame({
        "y_true": y_true,
        "y_pred": y_pred,
    })
    if df_meta is not None:
        pred_df["img_id"] = df_meta["img_id"].values
        pred_df["image_path"] = df_meta["image_path"].values
    if y_prob is not None:
        for i, name in enumerate(label_names):
            pred_df[f"prob_{name}"] = y_prob[:, i]
    pred_df.to_csv(out_dir / f"pred_{split_name}.csv", index=False)

    return summary, report


In [ ]:
labels = sorted(id_to_name.keys())
label_names = [id_to_name[i] for i in labels]

train_summary, _ = save_split_outputs("train", y_train, train_pred, labels, label_names, OUT_DIR, train_prob, train_df)
val_summary, _ = save_split_outputs("val", y_val, val_pred, labels, label_names, OUT_DIR, val_prob, val_df)
test_summary, _ = save_split_outputs("test", y_test, test_pred, labels, label_names, OUT_DIR, test_prob, test_df)

print("Train summary:")
print(json.dumps(train_summary, indent=2))
print("Val summary:")
print(json.dumps(val_summary, indent=2))
print("Test summary:")
print(json.dumps(test_summary, indent=2))

val_cm = confusion_matrix(y_val, val_pred, labels=labels)
test_cm = confusion_matrix(y_test, test_pred, labels=labels)
np.save(OUT_DIR / "confusion_val.npy", val_cm)
np.save(OUT_DIR / "confusion_test.npy", test_cm)

import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

for split_name, cm in [("val", val_cm), ("test", test_cm)]:
    fig, ax = plt.subplots(figsize=(8, 6))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_names)
    disp.plot(include_values=False, cmap="Blues", ax=ax, xticks_rotation=90)
    plt.title(f"{split_name.upper()} Confusion Matrix (CLIP + LR)")
    plt.tight_layout()
    fig_path = OUT_DIR / f"confusion_{split_name}.png"
    plt.savefig(fig_path, dpi=200)
    plt.close(fig)

from datetime import datetime, timezone
RUN_TIMESTAMP_UTC = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
RUN_ID = os.getenv("RUN_ID") or f"{OUT_DIR.name}_{RUN_TIMESTAMP_UTC.replace(':', '').replace('-', '')}"

run_meta = {
    "model": "clip_linear_baseline",
    "seed": SEED,
    "model_name": MODEL_NAME,
    "split_hash": (DATA_ROOT / "0_dataset_prep" / "out" / "metadata" / "split_hash.txt").read_text().strip()
        if (DATA_ROOT / "0_dataset_prep" / "out" / "metadata" / "split_hash.txt").exists() else None,
    "run_id": RUN_ID,
    "timestamp_utc": RUN_TIMESTAMP_UTC,
    "out_dir": str(OUT_DIR),
    "notebook_path": str(Path.cwd()),

}
with open(OUT_DIR / "run_meta.json", "w") as f:
    json.dump(run_meta, f, indent=2)

print("Saved outputs to", OUT_DIR)




Train summary:
{
  "accuracy": 0.7811742992271311,
  "balanced_accuracy": 0.8023489954042223,
  "macro_f1": 0.7674555686072319,
  "weighted_f1": 0.7850521226154938,
  "qwk": 0.8355813812929886,
  "mae": 0.24397277656015687,
  "rmse": 0.5513226161350039,
  "spearman": 0.7933401958674636,
  "auroc_ovr": 0.9412303242387868,
  "ece": 0.007901428268823339
}
Val summary:
{
  "accuracy": 0.7144408251900108,
  "balanced_accuracy": 0.6606546146466877,
  "macro_f1": 0.653242605100865,
  "weighted_f1": 0.7177184148463839,
  "qwk": 0.7989590586651415,
  "mae": 0.31704668838219324,
  "rmse": 0.6234644546140619,
  "spearman": 0.7511073702022578,
  "auroc_ovr": 0.8895017887658688,
  "ece": 0.08473441512942864
}
Test summary:
{
  "accuracy": 0.6791221826809015,
  "balanced_accuracy": 0.635708821036115,
  "macro_f1": 0.6020160353154397,
  "weighted_f1": 0.6890959860382733,
  "qwk": 0.7455024845658059,
  "mae": 0.36773428232502964,
  "rmse": 0.6871123408297379,
  "spearman": 0.7220317526470705,
  "auroc